# Session 5: Logistic Regression from Scratch with Gradient Descent

## Overview

Part 2 is about **classification**: predicting a category rather than a number. The first problem is a serious
one. A fine-needle aspirate (FNA) takes a few cells from a breast lump; a pathologist digitises the image and
software measures the cell nuclei. From those measurements, can a model say whether the lump is **malignant** or
**benign**, and how sure is it?

As in Session 1, you build the model by hand: **logistic regression**, fitted by gradient descent. You will see
why it needs a different cost from linear regression, why letting it train for ever makes it *worse*, how
**regularisation** fixes that, and how to choose a decision threshold when the two possible mistakes have very
different costs.

By the end, you will be able to:

- Explain the sigmoid function and why logistic regression outputs a probability.
- Write the log-loss cost and its gradient, and fit the model with gradient descent in NumPy.
- Draw and interpret a decision boundary.
- Recognise overfitting during training, and add an L2 penalty to prevent it.
- Match a hand-built model to scikit-learn's `LogisticRegression`, including its `C` parameter.
- Evaluate a classifier with a confusion matrix, precision and recall, and choose a threshold.

**Prerequisites:** Session 1 (gradient descent, scaling).

**Dataset:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).
569 FNA samples from the University of Wisconsin Hospitals. For ten properties of the cell nuclei (radius, texture,
perimeter, area, smoothness, compactness, concavity, concave points, symmetry and fractal dimension), the image
gives the **mean**, the **standard error** and the **worst** (mean of the three largest) value: 30 measurements in
all, and the diagnosis. It is downloaded automatically by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers **← you are here** |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** Can a hand-built logistic regression separate malignant from benign samples, and
how should it decide when it is unsure?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| numpy | The model, cost and gradients, by hand |
| pandas | Tables |
| scikit-learn | Train/test split, the reference model, metrics |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, precision_score, recall_score
from sklearn.model_selection import train_test_split

from importlib.metadata import version

for package in ["ucimlrepo", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data and split it

The columns come as `radius1`, `radius2`, `radius3` and so on; we rename the suffixes to `_mean`, `_se` and
`_worst`. The target is 1 for malignant (M) and 0 for benign (B).

`stratify=y` keeps the same share of malignant samples in the training and test sets. With only 569 samples, an
unlucky random split could otherwise put noticeably more malignant cases in one of them.

In [ ]:
from ucimlrepo import fetch_ucirepo

samples = fetch_ucirepo(id=17).data.original
samples.columns = [c.replace("1", "_mean").replace("2", "_se").replace("3", "_worst") for c in samples.columns]
X = samples.drop(columns=["ID", "Diagnosis"])
y = (samples["Diagnosis"] == "M").astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"{len(X)} samples, {X.shape[1]} measurements; {y.mean():.1%} malignant")
print(f"train: {len(X_train)} ({y_train.mean():.1%} malignant), test: {len(X_test)} ({y_test.mean():.1%} malignant)")
print(X_train[["radius_mean", "radius_se", "radius_worst", "concave_points_worst"]].describe().round(3).T)

**Read the output:** 569 samples, 37.3% malignant; 455 for training, 114 held out, with the same share of
malignant cases in both. The measurements are on very different scales (a radius of about 14 against concave
points around 0.1), so we will standardise them, exactly as in Session 1.

## Step 3: Look at two measurements

Start with two measurements so we can see everything on one chart: the **worst radius** (the size of the largest
nuclei) and the **worst concave points** (how many indentations their outlines have). We standardise them with the
training mean and standard deviation.

In [ ]:
mean, std = X_train.mean(), X_train.std(ddof=0)
Z_train = ((X_train - mean) / std).to_numpy()
Z_test = ((X_test - mean) / std).to_numpy()

TWO = ["radius_worst", "concave_points_worst"]
cols = [X.columns.get_loc(c) for c in TWO]
Z2_train, Z2_test = Z_train[:, cols], Z_test[:, cols]

fig, ax = plt.subplots(figsize=(6, 5))
for label, colour, name in [(0, "tab:blue", "benign"), (1, "tab:red", "malignant")]:
    ax.scatter(*Z2_train[y_train == label].T, s=12, alpha=0.6, c=colour, label=name)
ax.set_xlabel("worst radius (standardised)")
ax.set_ylabel("worst concave points (standardised)")
ax.legend()
plt.show()

**Read the output:** malignant samples (red) sit to the upper right: larger nuclei with more irregular outlines.
The two groups overlap only a little. A straight line through the middle would separate most of them; logistic
regression finds the best such line.

## Step 4: From a score to a probability: the sigmoid

Logistic regression first computes a linear score, exactly like linear regression:
$z = w_1 x_1 + w_2 x_2 + \dots + b$. That score can be any number, but we want a probability between 0 and 1. The
**sigmoid** (logistic) function squashes any number into that range:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

A score of 0 gives p = 0.5; large positive scores approach 1, large negative scores approach 0. The model
predicts "malignant" when p is above a **threshold**, usually 0.5, which is the same as z > 0. The line where
z = 0 is the **decision boundary**.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


for z in [-5, -2, 0, 2, 5]:
    print(f"z = {z:+d}  ->  p = {sigmoid(z):.3f}")

## Step 5: The cost: log-loss

For classification, the squared error is a poor cost: with the sigmoid inside it, the cost surface has flat
regions where gradient descent stalls. Instead we use the **log-loss** (cross-entropy):

$$J(w, b) = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i \log p_i + (1 - y_i)\log(1 - p_i)\Big]$$

For a malignant sample (y = 1) only the first term counts: the cost is $-\log p$, small when p is near 1 and huge
when the model confidently says "benign". For a benign sample it is $-\log(1-p)$. **Confident mistakes are
punished hardest.** The log-loss is *convex*: it has one bottom and no flat traps.

In [ ]:
def cost(Z, y, w, b):
    p = np.clip(sigmoid(Z @ w + b), 1e-12, 1 - 1e-12)      # avoid log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"malignant sample, model says p = {p:<4}  ->  cost {-np.log(p):.2f}")

**Read the output:** saying 0.99 for a truly malignant sample costs almost nothing (0.01). Saying 0.5 costs 0.69,
which is what a model that knows nothing pays on every sample. Saying 0.01, confidently wrong, costs 4.61.

## Step 6: The gradient and gradient descent

The derivative of the log-loss through the sigmoid simplifies to a remarkably clean form:

$$\frac{\partial J}{\partial w} = \frac{1}{n} X^\top (p - y) \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i)$$

It has the same shape as linear regression's gradient (prediction minus truth, times the input); only the
prediction p is different. Written with matrices, one line handles any number of inputs.

The `l2` argument is used from Step 9; leave it at 0 for now.

In [ ]:
def gradient_descent(Z, y, learning_rate=0.5, n_steps=1000, l2=0.0):
    w, b = np.zeros(Z.shape[1]), 0.0
    history = []
    for _ in range(n_steps):
        p = sigmoid(Z @ w + b)
        grad_w = Z.T @ (p - y) / len(y) + l2 * w
        grad_b = np.mean(p - y)
        w, b = w - learning_rate * grad_w, b - learning_rate * grad_b
        history.append(cost(Z, y, w, b))
    return w, b, history


w2, b2, history = gradient_descent(Z2_train, y_train, n_steps=2000)
print(f"weights: radius_worst {w2[0]:.3f}, concave_points_worst {w2[1]:.3f}; intercept {b2:.3f}")
print(f"training log-loss: start {cost(Z2_train, y_train, np.zeros(2), 0):.3f}, end {history[-1]:.4f}")

**Read the output:** the cost starts at 0.693 (the "knows nothing" value from Step 5, since all weights start at
0 and every p is 0.5) and falls to 0.134. Both weights are positive: bigger nuclei and more concave points both
raise the probability of malignancy.

## Step 7: Check against scikit-learn and draw the decision boundary

scikit-learn's `LogisticRegression` adds a penalty by default; `penalty=None` turns it off so it minimises
exactly the same cost as our loop.

In [ ]:
sk2 = LogisticRegression(penalty=None).fit(Z2_train, y_train)
print(f"gradient descent: w = {np.round(w2, 3)}, b = {b2:.3f}")
print(f"scikit-learn:     w = {np.round(sk2.coef_[0], 3)}, b = {sk2.intercept_[0]:.3f}")

p2_test = sigmoid(Z2_test @ w2 + b2)
print(f"test accuracy with two measurements: {accuracy_score(y_test, p2_test > 0.5):.3f}")

grid = np.linspace(-2.5, 4, 200)
xx, yy = np.meshgrid(grid, grid)
prob = sigmoid(w2[0] * xx + w2[1] * yy + b2)
fig, ax = plt.subplots(figsize=(6, 5))
contour = ax.contourf(xx, yy, prob, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.4)
ax.contour(xx, yy, prob, levels=[0.5], colors="black")
for label, colour in [(0, "tab:blue"), (1, "tab:red")]:
    ax.scatter(*Z2_test[y_test == label].T, s=14, c=colour, edgecolor="white", linewidth=0.5)
fig.colorbar(contour, label="P(malignant)")
ax.set_xlabel("worst radius (standardised)")
ax.set_ylabel("worst concave points (standardised)")
ax.set_title("Decision boundary (black) and test samples")
plt.show()

**Read the output:** the hand-built weights match scikit-learn's to within 0.02 (4.413 against 4.430, and 2.725
against 2.729); 2,000 more steps would close the gap further. On the 114 test samples, two measurements alone
classify 94.7% correctly.

In the chart, the black line is the decision boundary (p = 0.5), and the background colour shows the predicted
probability. Far from the line the model is confident; near it, probabilities are close to 0.5 and the few
mistakes happen there.

## Step 8: All 30 measurements, and the danger of training too long

Now use all 30 measurements and watch what happens as gradient descent runs for longer and longer.

In [ ]:
rows = []
for n_steps in [100, 1000, 10000, 50000]:
    w, b, history = gradient_descent(Z_train, y_train, n_steps=n_steps)
    p_test = sigmoid(Z_test @ w + b)
    rows.append({"steps": n_steps, "training log-loss": history[-1], "largest |weight|": np.abs(w).max(),
                 "test log-loss": log_loss(y_test, p_test), "test accuracy": accuracy_score(y_test, p_test > 0.5)})
print(pd.DataFrame(rows).set_index("steps").round(3))

**Read the output:** this one takes a few seconds. The **training** log-loss falls with every extra step, from
0.069 to 0.018. But the **test** log-loss gets five times *worse* (0.073 to 0.382) and test accuracy drops from 98%
to 95%. Meanwhile the largest weight grows from 0.9 to 8.8.

What is happening: with 30 measurements, the training samples can be separated almost perfectly. Gradient descent
can then always lower the training cost a little more by making the weights bigger, which pushes every training
probability closer to exactly 0 or 1. The model becomes **over-confident**, and on new samples near the boundary
its confident mistakes are punished hard. This is overfitting, here driven by the size of the weights.

## Step 9: Regularisation: penalise large weights

**L2 regularisation** adds a penalty for large weights to the cost:

$$J_{\text{reg}} = J + \frac{\lambda}{2}\sum_j w_j^2 \qquad\Rightarrow\qquad \frac{\partial J_{\text{reg}}}{\partial w} = \frac{\partial J}{\partial w} + \lambda w$$

The gradient gets one extra term, `l2 * w` in our function, which pulls every weight back towards zero. Now the
model only makes a weight large if that really pays off on the training data. The intercept is not penalised.

scikit-learn uses the same penalty, but expresses its strength as **C**, the *inverse*: smaller C means a
stronger penalty. With n training samples, $C = 1 / (\lambda n)$.

In [ ]:
l2 = 0.01
w_reg, b_reg, history = gradient_descent(Z_train, y_train, n_steps=50000, l2=l2)
p_test = sigmoid(Z_test @ w_reg + b_reg)
print(f"L2 = {l2}, 50,000 steps: largest |weight| = {np.abs(w_reg).max():.3f}, "
      f"test log-loss = {log_loss(y_test, p_test):.3f}, test accuracy = {accuracy_score(y_test, p_test > 0.5):.3f}")

sk = LogisticRegression(C=1 / (l2 * len(y_train)), max_iter=10000).fit(Z_train, y_train)
print(f"scikit-learn, C = {sk.C:.3f}: largest difference from our weights = "
      f"{np.abs(sk.coef_[0] - w_reg).max():.4f}")

**Read the output:** with the penalty, even 50,000 steps leave the weights small (largest 0.81) and the test
log-loss stays low (0.081, against 0.382 without it), with 98.2% test accuracy. Running longer no longer hurts: the penalty gives the
cost a proper bottom instead of an endless slope.

scikit-learn with the matching C = 0.22 finds the same weights, to within 0.003. This is why
scikit-learn regularises by default (C = 1.0): plain logistic regression on many inputs overfits easily.

## Step 10: How good is it? The confusion matrix

Accuracy hides *which* mistakes the model makes. A **confusion matrix** counts all four outcomes:

| | predicted benign | predicted malignant |
| --- | --- | --- |
| **actually benign** | true negative (TN) | false positive (FP): needless worry and biopsy |
| **actually malignant** | false negative (FN): a missed cancer | true positive (TP) |

Two summaries:

- **Recall** (sensitivity) = TP / (TP + FN): of the malignant samples, how many did we catch?
- **Precision** = TP / (TP + FP): of the samples we flagged, how many were really malignant?

In [ ]:
p_test = sigmoid(Z_test @ w_reg + b_reg)


def evaluate(threshold):
    pred = (p_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    return {"TN": tn, "FP": fp, "FN (missed)": fn, "TP": tp,
            "accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
            "recall": recall_score(y_test, pred)}


print(pd.DataFrame([evaluate(0.5)], index=["threshold 0.5"]).round(3).T)

**Read the output:** of the 42 malignant test samples, the model catches 40 (recall 0.952) and **misses 2**. It
raises no false alarm among the 72 benign samples (precision 1.0). Accuracy is 0.982.

## Step 11: Choosing the threshold

0.5 is a convention, not a law. A missed cancer is far worse than a false alarm (which leads to further tests), so
a screening tool should flag a sample at a **lower** probability. Lowering the threshold trades precision for
recall.

In [ ]:
table = pd.DataFrame({f"threshold {t}": evaluate(t) for t in [0.5, 0.3, 0.1, 0.05]}).T
print(table.round(3))

**Read the output:** lowering the threshold to 0.3 catches one more malignant sample at the cost of one false
alarm. At 0.1 the model misses **none** of the 42, at the price of 10 false alarms: precision falls to 0.81 and
accuracy to 0.91. Accuracy goes *down* while the tool gets safer, another reason not to judge a classifier by
accuracy alone. Which threshold is right
is not a statistical question: it depends on the cost of each kind of mistake, and should be decided with the
clinicians who will use the tool, *before* looking at the test results. Here we look at them only to show the
trade-off; in a real project the threshold is chosen on validation data.

A caution about these numbers: 114 test samples is few. One sample more or less is almost a percentage point of
accuracy. Session 9 uses cross-validation to get more stable comparisons.

## What this session hands to the next

- **Logistic regression from first principles:** sigmoid, log-loss, gradient, and the link to linear regression.
- **Regularisation:** why the default `LogisticRegression` has `C = 1.0`, and what changing C does.
- **Classification metrics:** confusion matrix, precision, recall, and the threshold as a separate decision.
- Session 6 builds a different kind of classifier by hand, **naive Bayes**, for text.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Using MSE as the cost for classification | Slow, can stall in flat regions | Use the log-loss |
| `np.log(0)` in the cost | `-inf` or `nan` | Clip probabilities to [1e-12, 1 - 1e-12] |
| Training an unregularised model for a long time | Growing weights, over-confident, worse test log-loss | Add an L2 penalty; in scikit-learn, keep the default or tune C |
| Thinking larger C means more regularisation | The opposite: C is the inverse strength | Smaller C = stronger penalty |
| Reporting accuracy only | Hides missed cancers | Show the confusion matrix, precision and recall |
| Choosing the threshold on the test set | Optimistic results | Choose it on validation data, from the cost of each error |
| Splitting a small dataset without `stratify` | Class shares differ between train and test | `train_test_split(..., stratify=y)` |

## Exercises

- [ ] In Step 6, try learning rates of 0.05 and 5. How many steps does each need to reach a cost of 0.14?
- [ ] Repeat Step 9 with `l2` = 0.001 and 0.1. How do the largest weight and the test log-loss change?
- [ ] Which three measurements have the largest weights in `w_reg`? Fit a model on just those three. How much
  worse is it than all 30?
- [ ] Plot recall against precision for thresholds from 0.01 to 0.99 (`sklearn.metrics.precision_recall_curve`
  does it for you).

Next, Session 6 builds a spam filter from scratch with naive Bayes, a classifier based on counting words and
applying Bayes' rule.